# Backpropagation Derivation

## 1. Network and Fixed Parameters

input 2개 → hidden neuron 2개 → output neuron 1개인 two-layer neural network를 다룬다. 두 layer 모두 Sigmoid activation function을 사용하며, target은 과제 예시의 $y=1$이다.

| Name | Value | Shape |
| --- | --- | --- |
| `x` | $\mathbf{x}=\begin{bmatrix}1 \\ 0\end{bmatrix}$ | $(2,1)$ |
| `W1` | $W^{(1)}=\begin{bmatrix}0.1 & 0.2 \\ 0.3 & 0.4\end{bmatrix}$ | $(2,2)$ |
| `b1` | $\mathbf{b}^{(1)}=\begin{bmatrix}0 \\ 0\end{bmatrix}$ | $(2,1)$ |
| `W2` | $W^{(2)}=\begin{bmatrix}0.5 & 0.6\end{bmatrix}$ | $(1,2)$ |
| `b2` | $b_1^{(2)}=0$ | scalar |
| `y_true` | $y=1$ | scalar |

$w_{ij}^{(l)}$은 이전 layer의 $j$번째 neuron에서 현재 layer의 $i$번째 neuron으로 연결되는 weight이다. $z$는 weighted sum, $a$는 activation output을 뜻한다.

$$
\sigma(t)=\frac{1}{1+e^{-t}}.
$$

중간 계산에는 반올림 전 값을 사용하고, NumPy 검증에서 소수점 4자리로 비교한다.

## 2. Forward Propagation by Hand

### Input → Hidden Layer: `z1`, `a1`

$$
\mathbf{z}^{(1)}=W^{(1)}\mathbf{x}+\mathbf{b}^{(1)}
=\begin{bmatrix}
0.1\times1+0.2\times0+0 \\
0.3\times1+0.4\times0+0
\end{bmatrix}
=\begin{bmatrix}0.1 \\ 0.3\end{bmatrix},
\qquad \text{shape: }(2,1).
$$

Sigmoid를 element-wise로 적용한다.

$$
\mathbf{a}^{(1)}=\sigma\!\left(\mathbf{z}^{(1)}\right)
=\begin{bmatrix}
(1+e^{-0.1})^{-1} \\
(1+e^{-0.3})^{-1}
\end{bmatrix}
\approx\begin{bmatrix}0.5249791875 \\ 0.5744425168\end{bmatrix},
\qquad \text{shape: }(2,1).
$$

### Hidden Layer → Output Layer: `z2`, `y_pred`, `loss`

$$
\begin{aligned}
z_1^{(2)}
&=w_{11}^{(2)}a_1^{(1)}+w_{12}^{(2)}a_2^{(1)}+b_1^{(2)} \\
&=0.5\sigma(0.1)+0.6\sigma(0.3)+0
\approx0.6071551038, \\
\hat{y}
&=\sigma\!\left(z_1^{(2)}\right)
=\frac{1}{1+e^{-z_1^{(2)}}}
\approx0.6472915700.
\end{aligned}
$$

`z2`와 `y_pred`는 scalar이다. NumPy에서는 $(1,2)\ @\ (2,1)$ matrix multiplication 결과의 유일한 element를 꺼내 `z2`로 사용한다.

Binary Cross-Entropy(BCE) loss도 scalar이며, $y=1$을 대입하면 다음과 같다.

$$
\begin{aligned}
L&=-\left[y\log\hat{y}+(1-y)\log(1-\hat{y})\right] \\
&=-\log\hat{y}\approx0.4349584369.
\end{aligned}
$$

## 3. Backpropagation Using the Chain Rule

loss에서 output layer, hidden layer 순서로 chain rule을 적용한다. $\odot$는 element-wise product이며, 각 gradient는 해당 variable과 같은 shape이다.

### Step 1. $\partial L/\partial\hat{y}$

$$
\begin{aligned}
\frac{\partial L}{\partial\hat{y}}
&=-\frac{y}{\hat{y}}+\frac{1-y}{1-\hat{y}} \\
&=\frac{\hat{y}-y}{\hat{y}(1-\hat{y})}.
\end{aligned}
$$

$y=1$을 대입하면 $\partial L/\partial\hat{y}=-1/\hat{y}\approx-1.5448988468$이며, scalar이다.

### Step 2. $\partial L/\partial z_1^{(2)}$ (Output Layer Checkpoint)

Sigmoid의 derivative는 $\sigma'(t)=e^{-t}/(1+e^{-t})^2=\sigma(t)(1-\sigma(t))$이다.

$$
\begin{aligned}
\delta_1^{(2)}:=\frac{\partial L}{\partial z_1^{(2)}}
&=\frac{\partial L}{\partial\hat{y}}
\frac{\partial\hat{y}}{\partial z_1^{(2)}} \\
&=\frac{\hat{y}-y}{\hat{y}(1-\hat{y})}\,\hat{y}(1-\hat{y}) \\
&=\boxed{\hat{y}-y}.
\end{aligned}
$$

$\delta_1^{(2)}=0.6472915700-1\approx-0.3527084300$이며, scalar이다.

### Step 3. $\partial L/\partial W^{(2)}$

output layer의 weighted sum에서 $\partial z_1^{(2)}/\partial w_{1i}^{(2)}=a_i^{(1)}$이므로,

$$
\frac{\partial L}{\partial w_{1i}^{(2)}}
=\frac{\partial L}{\partial z_1^{(2)}}
\frac{\partial z_1^{(2)}}{\partial w_{1i}^{(2)}}
=\delta_1^{(2)}a_i^{(1)},\qquad i=1,2.
$$

$$
\frac{\partial L}{\partial W^{(2)}}
=\delta_1^{(2)}\left(\mathbf{a}^{(1)}\right)^{\mathsf T}
\approx\begin{bmatrix}-0.1851645850 & -0.2026107182\end{bmatrix},
\qquad \text{shape: }(1,2).
$$

bias는 $\partial z_1^{(2)}/\partial b_1^{(2)}=1$이므로 $\partial L/\partial b_1^{(2)}=\delta_1^{(2)}\approx-0.3527084300$ (scalar)이다.

### Step 4. $\partial L/\partial\mathbf{a}^{(1)}$

output layer의 weighted sum에서 $\partial z_1^{(2)}/\partial a_i^{(1)}=w_{1i}^{(2)}$이므로,

$$
\frac{\partial L}{\partial a_i^{(1)}}
=\frac{\partial L}{\partial z_1^{(2)}}
\frac{\partial z_1^{(2)}}{\partial a_i^{(1)}}
=\delta_1^{(2)}w_{1i}^{(2)}.
$$

$$
\frac{\partial L}{\partial\mathbf{a}^{(1)}}
=\left(W^{(2)}\right)^{\mathsf T}\delta_1^{(2)}
\approx\begin{bmatrix}
0.5\times(-0.3527084300) \\
0.6\times(-0.3527084300)
\end{bmatrix}
\approx\begin{bmatrix}-0.1763542150 \\ -0.2116250580\end{bmatrix},
\qquad \text{shape: }(2,1).
$$

### Step 5. $\partial L/\partial\mathbf{z}^{(1)}$ (Hidden Layer Checkpoint)

각 hidden neuron에 자신의 Sigmoid derivative를 곱한다.

$$
\delta_i^{(1)}:=\frac{\partial L}{\partial z_i^{(1)}}
=\frac{\partial L}{\partial a_i^{(1)}}
\frac{\partial a_i^{(1)}}{\partial z_i^{(1)}}
=\frac{\partial L}{\partial a_i^{(1)}}a_i^{(1)}(1-a_i^{(1)}).
$$

$$
\begin{aligned}
\boldsymbol{\delta}^{(1)}
&=\frac{\partial L}{\partial\mathbf{a}^{(1)}}
\odot\mathbf{a}^{(1)}\odot\left(1-\mathbf{a}^{(1)}\right) \\
&\approx\begin{bmatrix}
-0.1763542150\times0.5249791875\times(1-0.5249791875) \\
-0.2116250580\times0.5744425168\times(1-0.5744425168)
\end{bmatrix} \\
&\approx\begin{bmatrix}-0.0439785158 \\ -0.0517335044\end{bmatrix},
\qquad \text{shape: }(2,1).
\end{aligned}
$$

### Step 6. $\partial L/\partial W^{(1)}$

$z_i^{(1)}=\sum_jw_{ij}^{(1)}x_j+b_i^{(1)}$에서 $\partial z_i^{(1)}/\partial w_{ij}^{(1)}=x_j$이므로,

$$
\frac{\partial L}{\partial w_{ij}^{(1)}}
=\frac{\partial L}{\partial z_i^{(1)}}
\frac{\partial z_i^{(1)}}{\partial w_{ij}^{(1)}}
=\delta_i^{(1)}x_j,\qquad i,j\in\{1,2\}.
$$

이를 outer product로 묶으면 네 weight의 gradient를 얻는다.

$$
\frac{\partial L}{\partial W^{(1)}}
=\boldsymbol{\delta}^{(1)}\mathbf{x}^{\mathsf T}
\approx\begin{bmatrix}
-0.0439785158 \\
-0.0517335044
\end{bmatrix}\begin{bmatrix}1 & 0\end{bmatrix}
=\begin{bmatrix}
-0.0439785158 & 0 \\
-0.0517335044 & 0
\end{bmatrix},
\qquad \text{shape: }(2,2).
$$

두 번째 column은 $x_2=0$이므로 모두 0이다. bias는 $\partial z_i^{(1)}/\partial b_i^{(1)}=1$이므로,

$$
\frac{\partial L}{\partial\mathbf{b}^{(1)}}
=\boldsymbol{\delta}^{(1)}
\approx\begin{bmatrix}-0.0439785158 \\ -0.0517335044\end{bmatrix},
\qquad \text{shape: }(2,1).
$$

## 4. NumPy Verification

같은 input으로 forward propagation과 backpropagation을 수행한 뒤, 위 손계산을 소수점 4자리로 반올림한 값 및 shape과 비교한다.

In [1]:
# Import NumPy for the forward and backward calculations.

import numpy as np

In [2]:
# Define the activation function used by both layers.

# Apply Sigmoid element-wise to a scalar or array.
def sigmoid(value):
    """Return the element-wise Sigmoid activation."""
    return 1.0 / (1.0 + np.exp(-value))

In [3]:
# Set the fixed input, parameters, and target from the hand calculation.

x = np.array([[1.0], [0.0]])
W1 = np.array([[0.1, 0.2], [0.3, 0.4]])
b1 = np.zeros((2, 1))
W2 = np.array([[0.5, 0.6]])
b2 = 0.0
y_true = 1.0

In [4]:
# Compute the forward intermediates and BCE loss without rounding.

z1 = W1 @ x + b1
a1 = sigmoid(z1)
# Extract the sole element of the (1, 1) matrix product as a scalar.
z2 = (W2 @ a1).item() + b2
y_pred = sigmoid(z2)
loss = -(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred))

In [5]:
# Apply the chain rule through the output layer.

d_y_pred = -y_true / y_pred + (1.0 - y_true) / (1.0 - y_pred)
d_z2 = d_y_pred * y_pred * (1.0 - y_pred)
d_W2 = d_z2 * a1.T
d_b2 = d_z2
# Transpose the output weights to send the scalar gradient to both hidden neurons.
d_a1 = W2.T * d_z2

In [6]:
# Propagate the gradient through the hidden activation to its weights and bias.

d_z1 = d_a1 * a1 * (1.0 - a1)
# The outer product restores the weight shape: (2, 1) @ (1, 2) -> (2, 2).
d_W1 = d_z1 @ x.T
d_b1 = d_z1.copy()

In [7]:
# Record the four-decimal hand-calculation references and their shapes.

hand_results = {
    "z1": np.array([[0.1000], [0.3000]]),
    "a1": np.array([[0.5250], [0.5744]]),
    "z2": 0.6072,
    "y_pred": 0.6473,
    "loss": 0.4350,
    "d_y_pred": -1.5449,
    "d_z2": -0.3527,
    "d_W2": np.array([[-0.1852, -0.2026]]),
    "d_b2": -0.3527,
    "d_a1": np.array([[-0.1764], [-0.2116]]),
    "d_z1": np.array([[-0.0440], [-0.0517]]),
    "d_W1": np.array([[-0.0440, 0.0000], [-0.0517, 0.0000]]),
    "d_b1": np.array([[-0.0440], [-0.0517]]),
}

In [8]:
# Compare all values and shapes with the hand calculations to four decimal places.

np.testing.assert_allclose(d_z2, y_pred - y_true, rtol=1e-12, atol=1e-14)
numpy_results = {
    "z1": z1, "a1": a1, "z2": z2, "y_pred": y_pred, "loss": loss,
    "d_y_pred": d_y_pred, "d_z2": d_z2, "d_W2": d_W2, "d_b2": d_b2,
    "d_a1": d_a1, "d_z1": d_z1, "d_W1": d_W1, "d_b1": d_b1,
}
for name, expected in hand_results.items():
    actual = np.round(numpy_results[name], decimals=4)
    np.testing.assert_array_equal(actual, expected, strict=True, err_msg=name)
    shape = np.shape(actual) or "scalar"
    # Add zero only for display to avoid printing negative zero.
    value = np.array2string(
        np.asarray(actual) + 0.0, precision=4, floatmode="fixed", separator=", "
    ).replace("\n", " ")
    print(f"{name} | shape={shape} | hand = NumPy = {value} | PASS")

print("All values and shapes match the hand calculations to 4 decimal places.")

z1 | shape=(2, 1) | hand = NumPy = [[0.1000],  [0.3000]] | PASS
a1 | shape=(2, 1) | hand = NumPy = [[0.5250],  [0.5744]] | PASS
z2 | shape=scalar | hand = NumPy = 0.6072 | PASS
y_pred | shape=scalar | hand = NumPy = 0.6473 | PASS
loss | shape=scalar | hand = NumPy = 0.4350 | PASS
d_y_pred | shape=scalar | hand = NumPy = -1.5449 | PASS
d_z2 | shape=scalar | hand = NumPy = -0.3527 | PASS
d_W2 | shape=(1, 2) | hand = NumPy = [[-0.1852, -0.2026]] | PASS
d_b2 | shape=scalar | hand = NumPy = -0.3527 | PASS
d_a1 | shape=(2, 1) | hand = NumPy = [[-0.1764],  [-0.2116]] | PASS
d_z1 | shape=(2, 1) | hand = NumPy = [[-0.0440],  [-0.0517]] | PASS
d_W1 | shape=(2, 2) | hand = NumPy = [[-0.0440,  0.0000],  [-0.0517,  0.0000]] | PASS
d_b1 | shape=(2, 1) | hand = NumPy = [[-0.0440],  [-0.0517]] | PASS
All values and shapes match the hand calculations to 4 decimal places.
